# Curate a new FDA indication using the tools only

This notebook demonstrates how to curate a newly identified indication from a newer FDA label for Verzenio (`NDA208716`) using only the public `moalmanac-fda-curation` command-line tools. Verzenio already exists in MOAlmanac, so this is the **new-indication branch of the update workflow**, not first-time document curation. It does not invoke the curation skill or rely on an agent to sequence commands or make decisions.

The notebook deliberately stops at curator gates. Review the generated Markdown and source evidence before changing a confirmation variable and recording a decision. Final update assembly also requires screening any revisions to existing Verzenio indications; that complementary path is shown in `revise-existing-indication-tool-only.ipynb`.

## 1. Configure the run

Prerequisites:

- install this project in the notebook kernel environment;
- set `ANTHROPIC_API_KEY` in `.env` or the environment;
- place a `moalmanac-db` checkout beside this repository (the commands verify it and name any missing input); and
- start with a new work directory, or intentionally reuse its existing artifacts.

For reproducibility, record the repository commits used when saving the completed run. FDA and database data can change over time.

In [ ]:
from pathlib import Path
from IPython.display import FileLink, display
import json
import subprocess
import sys

APPLICATION_NUMBER = "NDA208716"
REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "pyproject.toml").is_file():
    REPO_ROOT = REPO_ROOT.parent
DATABASE_DIR = REPO_ROOT.parent / "moalmanac-db"
WORK_DIR = REPO_ROOT / "outputs" / "tool-only-new-nda208716"

print(f"Application: {APPLICATION_NUMBER}")
print(f"Database: {DATABASE_DIR}")
print(f"Work directory: {WORK_DIR}")

In [ ]:
def run_tool(*arguments):
    command = [sys.executable, "-m", "moalmanac_fda_curation.cli", *map(str, arguments)]
    print("$", " ".join(command))
    return subprocess.run(command, cwd=REPO_ROOT, check=True, text=True)

def load_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))

def show_file(path):
    path = Path(path).resolve()
    if not path.is_file():
        raise FileNotFoundError(path)
    display(FileLink(str(path)))

def record_decision(stage, decision, indication_index=None, overrides=None, note=None):
    arguments = ["record-decision", "--work-dir", WORK_DIR, "--stage", stage, "--decision", decision]
    if indication_index is not None:
        arguments.extend(["--indication-index", indication_index])
    for field, value in (overrides or {}).items():
        arguments.extend(["--override", f"{field}={json.dumps(value)}"])
    if note:
        arguments.extend(["--note", note])
    return run_tool(*arguments)

## 2. Check setup and confirm the workflow route

The status check should report that Verzenio is already curated from the March 2023 label and that a newer approved label is available.

In [ ]:
run_tool("check-setup")

In [ ]:
status_path = WORK_DIR / "intermediate" / "curation-status.json"
if not status_path.exists():
    run_tool(
        "check-curation-status",
        "--application-number", APPLICATION_NUMBER,
        "--database-dir", DATABASE_DIR,
        "--output-json", status_path,
    )
status = load_json(status_path)
display(status)
if not status.get("previously_curated"):
    raise RuntimeError("NDA208716 is not present in this database checkout.")
if not status.get("newer_label_available"):
    raise RuntimeError("NDA208716 is curated, but no newer approved FDA label is available.")

## 3. Extract the newer label and reconcile its indications

`find-new-indications` prepares the newer-label document proposal, extracts its indications, and maps them against the existing Verzenio indications in `moalmanac-db`. It creates review files for newly identified indications and for `not_found` or `uncertain` mappings.

In [ ]:
matches_path = WORK_DIR / "intermediate" / "indication-matches.json"
if not matches_path.exists():
    run_tool(
        "find-new-indications",
        "--application-number", APPLICATION_NUMBER,
        "--database-dir", DATABASE_DIR,
        "--work-dir", WORK_DIR,
    )
matches = load_json(matches_path)
for mapping in matches.get("mappings", []):
    print(mapping.get("classification"), mapping.get("existing_indication_id"), mapping.get("latest_indication_index"), sep=" | ")
for path in sorted((WORK_DIR / "review").rglob("*.md")):
    show_file(path)

In [ ]:
mapping_exceptions = [
    item for item in matches.get("mappings", [])
    if item.get("classification") in {"not_found", "uncertain"}
]
CONFIRM_MAPPING_EXCEPTIONS_REVIEWED = False
if mapping_exceptions and not CONFIRM_MAPPING_EXCEPTIONS_REVIEWED:
    raise RuntimeError("Review and resolve every not_found or uncertain mapping before continuing.")

## 4. Screen newly identified indication candidates

The reconciliation retains complete source records for latest-label indications classified as `new`, then filters to biomarker-relevant curation candidates. Review `new-indications.md` against the newer label before selecting anything.

In [ ]:
indication_fields_path = next((WORK_DIR / "intermediate").glob("*-claude_chunked_indication_fields.json"))
indication_payload = load_json(indication_fields_path)
new_candidates = matches.get("new_indication_candidates", [])
for candidate in new_candidates:
    index = candidate["latest_indication_index"]
    item = candidate
    print(f"[{index}] {item.get('indication')}")
    print(f"    biomarker={item.get('raw_biomarkers')!r}")
    print(f"    cancer_types={item.get('raw_cancer_types')!r}")
    print(f"    therapeutics={item.get('raw_therapeutics')!r}")
if new_candidates:
    show_file(WORK_DIR / "review" / "new-indications.md")
else:
    print("The newer label contains no biomarker-relevant new indication candidates.")

Choose only the biomarker-relevant candidates that belong in MOAlmanac. Candidate indexes are positions in the latest-label extraction, not permanent MOAlmanac identifiers.

In [ ]:
# CURATOR GATE: populate after comparing every candidate with the label.
SELECTED_INDICATION_INDEXES = [3]
EXCLUDED_INDICATION_INDEXES = []
CONFIRM_CANDIDATE_DECISIONS = True

all_indexes = {item["latest_indication_index"] for item in new_candidates}
decided_indexes = set(SELECTED_INDICATION_INDEXES) | set(EXCLUDED_INDICATION_INDEXES)
if not CONFIRM_CANDIDATE_DECISIONS:
    raise RuntimeError("Review the candidates, populate both index lists, and confirm the decisions.")
if decided_indexes != all_indexes or set(SELECTED_INDICATION_INDEXES) & set(EXCLUDED_INDICATION_INDEXES):
    raise ValueError("Each candidate must appear exactly once in selected or excluded indexes.")
print(f"Selected for detailed review: {SELECTED_INDICATION_INDEXES}")
print(f"Excluded after screening: {EXCLUDED_INDICATION_INDEXES}")
for index in EXCLUDED_INDICATION_INDEXES:
    record_decision("indication", "excluded", indication_index=index)

## 5. Prepare detailed reviews for selected indications

The shared downstream workflow drafts a description, builds the historical Section 1 changelog, matches the indication to its earliest supporting label event, and generates indication, description, and approval review files.

In [ ]:
if not SELECTED_INDICATION_INDEXES:
    raise RuntimeError("No indication candidates were selected.")
arguments = ["prepare-selected-review", "--work-dir", WORK_DIR]
for index in SELECTED_INDICATION_INDEXES:
    arguments.extend(["--indication-index", index])
run_tool(*arguments)
for path in sorted((WORK_DIR / "review").rglob("*.md")):
    show_file(path)

## 6. Record indication, description, and approval decisions

Open the three generated Markdown files for each selected indication: `indication.md`, `description.md`, and `approval.md`. For each stage, use `accepted` when the proposal is correct as written. Use `edited` and place only the corrected fields in `overrides` when it needs a change. The approval review proposes both the initial approval date and the approval status, so a status correction is `{"status": "Accelerated"}`. After completing all three reviews, set `CONFIRM_DOWNSTREAM_DECISIONS = True` and run the cell.

In [ ]:
# CURATOR GATE: one block is required for every index in SELECTED_INDICATION_INDEXES.
# This Verzenio run selected index 3.
DOWNSTREAM_DECISIONS = {
    3: {
        "indication": {"decision": "accepted", "overrides": {}},
        "description": {"decision": "accepted", "overrides": {}},
        "approval": {"decision": "accepted", "overrides": {}},
    },
}
CONFIRM_DOWNSTREAM_DECISIONS = True

if not CONFIRM_DOWNSTREAM_DECISIONS:
    raise RuntimeError("Review indication.md, description.md, and approval.md, then confirm.")
if set(DOWNSTREAM_DECISIONS) != set(SELECTED_INDICATION_INDEXES):
    raise ValueError("Provide downstream decisions for every selected indication index.")
for index, stages in DOWNSTREAM_DECISIONS.items():
    for stage in ("indication", "description", "approval"):
        configuration = stages[stage]
        record_decision(
            stage,
            configuration["decision"],
            indication_index=index,
            overrides=configuration.get("overrides", {}),
        )

## 7. Continue to integrated update assembly

New indications and revisions to existing indications share one update artifact set. Run `revise-existing-indication-tool-only.ipynb` to screen changes to existing Verzenio indications and call `assemble-revisions`. That assembly step assigns new identifiers to accepted new indications and adds their curator and FDA approval contributions, suggests replacement records for changed indications accepted during revision screening, and writes the document and URL updates.

In [ ]:
print("New-indication decisions are recorded in:")
show_file(WORK_DIR / "review" / "decisions.json")
print("Complete revision screening before assembling the integrated update artifacts.")

## Handoff

This focused notebook completes the review decisions for newly identified Verzenio indications. Because document metadata, new indications, and revised indications are assembled together, final JSON materialization occurs after the complementary revision-screening workflow. Preserve this work directory with the review decisions and source artifacts so another curator can audit how the records were produced.